# nablatensor — GPU bench harness run (for the docs)

Runs the project's committed bench mains on a **Colab GPU runtime** (`cuda`,
plus `opencl` where a single engine is pinned) and prints their raw tables +
one plain-text report. The purpose is to produce the `cuda` numbers that
`docs/` and the `nablatensor-web` blog currently mark as unavailable.

Pick a GPU runtime first: *Runtime ▸ Change runtime type ▸ T4 GPU*.

In [ ]:
# --- Setup. On Colab: install JDK 25 + Maven, clone the branch, build,
#     install nothing extra (we run the bench mains straight off target/classes). ---
import os, sys, subprocess, shlex, time, textwrap

ON_COLAB = "google.colab" in sys.modules
BRANCH = "bench/t4-gpu"

if not ON_COLAB:
    PROJECT_ROOT = os.environ.get("NABLATENSOR_HOME") or os.getcwd()
else:
    PROJECT_ROOT = "/content/nablatensor"
    JDK_HOME = "/opt/jdk-25"
    if not os.path.isdir(PROJECT_ROOT + "/.git"):
        _script = r'''
            set -eux
            if [ ! -x "$JDK_HOME/bin/java" ]; then
                curl -fsSL -o /tmp/jdk25.tgz \
                  "https://api.adoptium.net/v3/binary/latest/25/ga/linux/x64/jdk/hotspot/normal/eclipse"
                mkdir -p "$JDK_HOME"; tar -xzf /tmp/jdk25.tgz -C "$JDK_HOME" --strip-components=1
            fi
            command -v mvn >/dev/null 2>&1 || { apt-get -qq update && apt-get -qq install -y maven; }
            [ -d "$PROJECT_ROOT/.git" ] || \
              git clone --depth 1 -b "$BRANCH" \
                https://github.com/nablatensor-dev/nablatensor.git "$PROJECT_ROOT"
            cd "$PROJECT_ROOT"
            JAVA_HOME="$JDK_HOME" MAVEN_OPTS=--sun-misc-unsafe-memory-access=allow \
              mvn -q -T1C -DskipTests install
        '''
        subprocess.run(["bash", "-c", _script], check=True,
                       env={**os.environ, "JDK_HOME": JDK_HOME,
                            "PROJECT_ROOT": PROJECT_ROOT, "BRANCH": BRANCH})
    os.environ["JAVA_HOME"] = JDK_HOME
    os.environ["PATH"] = JDK_HOME + "/bin:" + os.environ["PATH"]
    os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")

JAVA = os.path.join(os.environ.get("JAVA_HOME", "/usr"), "bin", "java")
import glob
CP = ":".join(sorted(glob.glob(os.path.join(PROJECT_ROOT, "*", "target", "classes"))))
assert CP, f"no target/classes under {PROJECT_ROOT}"
print("java   :", JAVA)
print("root   :", PROJECT_ROOT)
print("classes:", CP.count(":") + 1, "modules")

In [ ]:
# --- run a bench main straight off the classpath, capture its stdout ---
JVM_ARGS = ["--enable-native-access=ALL-UNNAMED", "--add-modules=jdk.incubator.vector"]

results = {}   # label -> stdout text

def run_main(label, main_class, props=None, timeout=1800):
    props = props or {}
    cmd = [JAVA, *JVM_ARGS, *(f"-D{k}={v}" for k, v in props.items()),
           "-cp", CP, main_class]
    print(f"\n===== {label} :: {main_class} {props} =====", flush=True)
    t = time.perf_counter()
    try:
        cp = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout,
                            cwd=PROJECT_ROOT)
        out = cp.stdout
        if cp.returncode != 0:
            out += f"\n[exit {cp.returncode}]\n" + "\n".join(cp.stderr.strip().splitlines()[-15:])
    except subprocess.TimeoutExpired:
        out = f"[timeout > {timeout}s]"
    dt = time.perf_counter() - t
    print(out)
    print(f"----- ({label}: {dt:.0f}s) -----", flush=True)
    results[label] = out
    return out

In [ ]:
# --- sanity: the GPU is visible, and a tiny run picks up cuda -------------
print(subprocess.run(
    ["bash", "-c", "nvidia-smi --query-gpu=name,memory.total,driver_version "
     "--format=csv,noheader || echo 'no nvidia-smi'"],
    capture_output=True, text=True).stdout)
run_main("probe (tiny Benchmarks)", "com.nablatensor.bench.Benchmarks",
         {"scenarios": 200_000, "steps": 8}, timeout=600)

In [ ]:
# --- the runs the docs need -------------------------------------------------

# 1. headline adjoint-vs-bump + backend matrix (README "The benchmark")
run_main("Benchmarks", "com.nablatensor.bench.Benchmarks",
         {"scenarios": 2_000_000, "steps": 252, "seed": 42})

# 2. per-product adjoint vs bump on cuda (docs/compare/vs-bump-and-revalue.md)
run_main("ProductBench cuda", "com.nablatensor.bench.ProductBench",
         {"scenarios": 1_000_000, "steps": 128, "seed": 42, "engine": "cuda"})

# 3. fp32 per-product, cuda then opencl (four-ways-to-talk-to-a-gpu fp32 matrix)
run_main("Fp32ProductBench cuda", "com.nablatensor.bench.Fp32ProductBench",
         {"scenarios": 1_000_000, "steps": 128, "seed": 42, "engine": "cuda"})
run_main("Fp32ProductBench opencl", "com.nablatensor.bench.Fp32ProductBench",
         {"scenarios": 1_000_000, "steps": 128, "seed": 42, "engine": "opencl"})

# 4. FRTB curvature backend table (docs/examples/frtb-curvature-showcase.md)
#    self-iterates {cpu, cpu-jit, simd, cuda}
run_main("CurvatureBackendRun", "com.nablatensor.bench.CurvatureBackendRun",
         {"scenarios": 1_000_000, "steps": 252, "seed": 42, "rounds": 3})

# 5. Asian risk run fp64 + fp32 matrices + 1e10 projection (docs/examples/asian-greeks.md)
#    self-iterates available engines for each precision
run_main("AsianRiskRun", "com.nablatensor.bench.AsianRiskRun",
         {"probe": 1_000_000, "steps": 252, "seed": 42})

In [ ]:
# --- one plain-ASCII report -------------------------------------------------
import datetime, platform

def _sh(c):
    try:
        return subprocess.run(c, shell=True, capture_output=True, text=True, timeout=15).stdout.strip()
    except Exception:
        return ""

gpu  = _sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") or "none"
cpu  = _sh("grep -m1 'model name' /proc/cpuinfo | cut -d: -f2 | sed 's/^ //'") or "unknown"
jver = _sh(f'"{JAVA}" -version 2>&1 | head -1')
sha  = _sh(f"git -C '{PROJECT_ROOT}' rev-parse --short HEAD")
brn  = _sh(f"git -C '{PROJECT_ROOT}' rev-parse --abbrev-ref HEAD")

L = ["=" * 100,
     "NABLATENSOR — GPU BENCH HARNESS RUN",
     "=" * 100,
     f"generated  : {datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M:%SZ}",
     f"host       : {platform.platform()}",
     f"cpu        : {cpu}  x{os.cpu_count()}",
     f"gpu        : {gpu}",
     f"java       : {jver}",
     f"nablatensor: {sha} ({brn})",
     f"colab      : {ON_COLAB}",
     "=" * 100]
for label, out in results.items():
    L += ["", "#" * 100, f"# {label}", "#" * 100, "", out.rstrip()]
L += ["", "=" * 100]

report = "\n".join(L)
outp = "/content/nablatensor-gpu-bench.txt" if ON_COLAB else "nablatensor-gpu-bench.txt"
open(outp, "w").write(report + "\n")
print("----- COPY BELOW " + "-" * 82)
print(report)
print("----- COPY ABOVE " + "-" * 82)
print(f"\n(also written to {outp})")